# Iteration 7 — Image-only, curvature-capable pia via patch-MAX + TPS

**Goal.** After the user audit of iter 6, two specific failures remained: (a) on **755252**, the N22 (centroid-based) surface *under-estimates* the real pia — the true surface is deeper, but v2.1/v2.2 are planar quadratics that cannot track the curvature.  (b) On **790322**, iter-6 auto sits too deep (v3 sub-iter landed +94 µm below auto).  Both failures point at the IRLS-quadratic fit: the real pia is curved, so an arbitrary planar-quadratic surface cannot match it.

**Approach.** Replace the quadratic with a regularised thin-plate-spline via `scipy.interpolate.RBFInterpolator` (kernel=`thin_plate_spline`) over a 20 × 20 interior grid of per-column OOT→tissue transition points.  The detector question then becomes: *how do we find the transition z in each column robustly, from pixel values alone*?  Six sub-iterations below converge on one answer.

**Success criterion.** TPS surface (i) sits below N22 on 755252 (matching user's qualitative read), (ii) sits close to auto on 790322 (closing the gap), (iii) curvature is visible in XZ / YZ overlays, (iv) stays consistent on the other 4 subjects.

## Sub-iteration history

Each row feeds a single thin-plate-spline surface on the 20 × 20 interior-padded grid; the difference is in how the per-column OOT→tissue transition z is detected.  Median z is reported for the two prototype subjects.

| sub-iter | detector | 755252 z_um | 790322 z_um | verdict |
|---|---|---:|---:|---|
| v1 | global log-combined Gaussian LLR | 416 | 192 | LLR dragged down by L1 faint neuropil |
| v2 | 488 col-local mean+k·std (MAD) | 88 / noise-floor catch | 128 / only 22 valid | 488 lacks L1 contrast; caught noise floor rise |
| v3 | abs log-combined thr = −4 + sustain 15 µm | 404 (+28 vs N22) ✓ | 232 (+94 vs auto) ✗ | good on cliff-subject 755252, fails on gradient-subject 790322 |
| v4 | v3 ∪ first log-combined peak (h=−2, prom=2.5) | 404 | 232 | peaks land on deeper L2 cells, not early sparse L1 cells |
| v5 | col-adaptive thr = max(−6.3, col_p90 − 3.0) | 412 | 236 | per-col p90 is dominated by whichever cell spike each col has → thr ≈ const ≈ −4 regardless of subject |
| v5 + patch-mean (15×15 vox ≈ 60 µm) | same thr on log-mean of patch | 404 (+26) | 172 (+30 vs auto) | averaging voxels smears sparse cells |
| **v6 = v5 + patch-MAX (15×15 vox)** | same col-adaptive thr on log-max of patch | **400 (+25.9 vs N22)** | **152 (+4.4 vs auto)** | **patch log-max preserves sparse bright cells → surface tracks the earliest tissue in each neighborhood** |

**Key insight** (v6): taking **log-MAX over a 15 × 15 xy patch** per grid point — instead of log-mean of a single voxel — is what closes the gap on subjects with sparse L1 cells.  Log of the mean dilutes isolated bright cells; max preserves them.  For cliff subjects (755252) patch max ≡ patch mean once tissue is dense, so the result is unchanged.

## Locked parameters (v6)

```python
EPS = 1e-3
SMOOTH_Z_UM = 5.0       # Gaussian z-smoothing before threshold test
SUSTAIN_Z_UM = 15.0     # must stay above thr for this span to fire
DELTA_LOG = 3.0         # col-adaptive thr = col_p90_log − DELTA_LOG
THR_FLOOR = -6.3        # never fire below this (log(eps) = −6.9)
PATCH_W = 7             # half-width in voxels → 15×15 xy ≈ 60 µm at level 4
TPS_SMOOTHING = 200.0   # RBFInterpolator regularisation
DETECTION_SOURCE = "combined"   # log of bg-subtracted combined volume
```

All other knobs (N_SIDE=20, EDGE_FRAC=0.15) inherit the scoring-grid conventions used throughout session 03c.

## Final 6-subject HCR bench

In [ ]:
from pathlib import Path
import pandas as pd

SESSION = Path('/root/capsule/code/sessions/03c_onset_features')
df = pd.read_csv(SESSION / 'data' / 'iter07_summary.csv')
df['subject'] = df.subject.astype(str)
cols = ['subject', 'auto_selected', 'detection_source', 'median_thr',
        'n_valid_trans', 'median_trans_z_um',
        'delta_z_vs_n22_um', 'delta_z_vs_auto_um']
print(df[cols].to_string(index=False, float_format=lambda x: f'{x:7.2f}'))

**How to read the deltas.**  `delta_z_vs_n22_um` is `median(z_iter07 − z_N22)` on the grid; positive = iter07 below N22 (deeper).  `delta_z_vs_auto_um` is the analogous comparison to the iter-6 auto-selected v2.2 surface.

- All 6 subjects land **4–29 µm below N22**, consistently.  N22 is a slight-shallow-biased estimator on this bench; iter07 is always deeper.
- **755252** (`+25.9 µm` vs N22): matches the user's qualitative finding ("real surface is below N22").  Also +26.2 vs v2.1/v2.2 auto, which were both quadratic.
- **790322** (`+4.4 µm` vs auto): the patch-MAX fix collapsed the v3 gap (+94 µm) to near-zero.  Still +25.6 below N22 like the rest.
- Valid-transition count is 400/400 on every subject — no OOT drops, no detector failures.

## Surface overlays (XZ / YZ on combined + log(combined))

Each subject: 2 × 2 panels showing the XZ mid-Y and YZ mid-X slices of the combined HCR volume (top row: raw, bottom-left / bottom-right: log).  Four surfaces overlaid:

- **green** — N22 centroid-based quantile ceiling (the previous "truth").
- **orange** — v2.1 default image-based IRLS quadratic.
- **cyan** — v2.2 auto (`estimate_pia_surface_image_autoselect`, iter-4/6 candidate-bank).
- **red** — iter07 TPS (patch-MAX detector, this iteration).

Figure title quotes `median_trans_z`, `median_thr`, and valid-count from the per-column transition detector.

In [ ]:
from IPython.display import Image, display, Markdown

subjects = ['755252', '767018', '767022', '782149', '788406', '790322']
for sid in subjects:
    p = SESSION / 'figures' / f'iter07_surfaces_{sid}.png'
    if p.exists():
        display(Markdown(f'### {sid}'))
        display(Image(filename=str(p)))

## Transition diagnostics

Per subject, 3 panels:

1. Histogram of per-column transition z (µm) across the 20 × 20 grid — spread tells us how curved the detected surface is.
2. Histogram of per-column adaptive log-thresholds `max(THR_FLOOR, col_p90 − DELTA_LOG)` — reveals whether the subject is cliff-type (thr clustered near −4) or gradient-type (thr lower, near the floor).
3. Smoothed log-combined column profiles at 5 sampled grid columns with their per-column thresholds — lets you eyeball how the adaptive threshold cuts across OOT vs tissue on each column.

In [ ]:
for sid in subjects:
    p = SESSION / 'figures' / f'iter07_dists_{sid}.png'
    if p.exists():
        display(Markdown(f'### {sid}'))
        display(Image(filename=str(p)))

## Bottom line

- **Patch log-MAX + column-adaptive log threshold + TPS** is the first image-only pia estimator that is (i) curvature-capable and (ii) consistent across cliff-type subjects (755252) AND gradient-type subjects (790322).  The two long-standing failure modes of v2.1/v2.2 (under-fit on 755252 curvature; over-deep on 790322) are both gone.
- Surface lands 4–29 µm below N22 on all 6 subjects — confirming the user's observation that N22 is a mildly shallow-biased truth for HCR.  If downstream applications want a "pia surface in the image" rather than "centroid ceiling", iter07 is the better anchor.
- **Why patch-MAX works:** a single voxel column on a grid point is dominated by whichever voxel happens to land there; averaging a 15 × 15 xy patch smears sparse L1 cells into the background; MAX over the same patch preserves the earliest bright cell in a 60-µm neighborhood.  Combined with a per-column p90-based threshold, the detector fires at "first meaningful tissue signal" regardless of whether the column is cliff-type or gradient-type.
- **Why TPS works:** a 20 × 20 grid of per-column transition points is the right amount of data for a regularised thin-plate-spline — enough to resolve the mm-scale pia curvature, few enough that the `smoothing=200` regulariser can absorb outlier columns without deforming the surface.
- **Next step.** Decide whether to promote iter07 as the default in `estimate_pia_surface_image_autoselect` (replacing the quadratic IRLS candidate bank entirely), or keep it as a parallel `_tps` entry point until more subjects (mouse GFP+, E15.5 cortex, non-HCR modalities) validate the patch-MAX detector outside this benchmark.